# COMP6713 — Notebook 1: Data Exploration

This notebook explores the three medical QA datasets used in this project:
- **PubMedQA** (1,000 labelled pairs) — biomedical yes/no/maybe QA from PubMed abstracts  
- **BioASQ** (8,216 pairs) — factoid biomedical QA  
- **MedQA-USMLE** (11,451 pairs) — US medical licensing exam questions  

All datasets load directly from HuggingFace — no account required.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
from collections import Counter
from medqa.data.loader import load_pubmedqa, load_bioasq, load_medqa_usmle, load_all

## 1. Load Datasets

In [ ]:
pubmedqa = load_pubmedqa()
bioasq   = load_bioasq()
usmle    = load_medqa_usmle()

print(f"PubMedQA : {len(pubmedqa):>6} records")
print(f"BioASQ   : {len(bioasq):>6} records")
print(f"USMLE    : {len(usmle):>6} records")
print(f"Total    : {len(pubmedqa)+len(bioasq)+len(usmle):>6} records")

## 2. Sample Records

In [ ]:
# PubMedQA sample
r = pubmedqa[0]
print("=== PubMedQA Sample ===")
print(f"Question : {r['question']}")
print(f"Answer   : {r['answer'][:200]}")
print(f"Type     : {r['answer_type']}")
print(f"Context  : {r['context'][:300]}...")

In [ ]:
# BioASQ sample
r = bioasq[0]
print("=== BioASQ Sample ===")
print(f"Question : {r['question']}")
print(f"Answer   : {r['answer']}")
print(f"Context  : {r['context'][:300]}...")

In [ ]:
# MedQA-USMLE sample
r = usmle[0]
print("=== MedQA-USMLE Sample ===")
print(f"Question : {r['question']}")
print(f"Options  : {r['context']}")
print(f"Answer   : {r['answer']}")

## 3. Dataset Statistics

In [ ]:
all_records = load_all()
df = pd.DataFrame(all_records)

print("Dataset distribution:")
print(df['source'].value_counts())
print(f"\nTotal labelled records: {len(df)}")

In [ ]:
# Question length distribution
df['q_len'] = df['question'].str.len()
df['ctx_len'] = df['context'].str.len()
df['ans_len'] = df['answer'].str.len()

print("Question length (chars):")
print(df.groupby('source')['q_len'].describe()[['mean','min','max']].round(1))
print("\nContext length (chars):")
print(df.groupby('source')['ctx_len'].describe()[['mean','min','max']].round(1))

In [ ]:
# PubMedQA answer type distribution
pqa_df = pd.DataFrame(pubmedqa)
print("PubMedQA answer types:")
print(pqa_df['answer_type'].value_counts())

## 4. Train / Test Split

In [ ]:
from medqa.data.preprocessor import split_dataset
train, test = split_dataset(all_records)
print(f"Train set : {len(train)} records ({len(train)/len(all_records)*100:.1f}%)")
print(f"Test set  : {len(test)} records ({len(test)/len(all_records)*100:.1f}%)")

## Summary

| Dataset | Records | Type | Source |
|---------|---------|------|--------|
| PubMedQA | 1,000 | Yes/No/Maybe QA | PubMed abstracts |
| BioASQ | 8,216 | Factoid QA | Biomedical literature |
| MedQA-USMLE | 11,451 | Multiple choice | US medical licensing exam |
| **Total** | **20,667** | | |

The combined dataset of ~20k records is used for training and evaluation.
An additional ~61k unlabelled PubMedQA records are used to enrich the RAG vector store.